# Otimização — capítulo 5: Otimização global
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/notebooks/05_global_optimization.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 5 of the course "Optimization" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/optimization-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
if os.path.isfile(os.path.join("..", "uc_setup.py")):
    RAIZ = os.path.abspath("..")
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 5.1 GLODS — Global and Local Optimization using Direct Search

Código de investigação do docente, distribuído na aula; não faz parte do código da UC nem deste caderno. Ver [`README.md`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_1_glods/README.md).

## 5.2 Simulated annealing

Exemplo [`ex05_2_simulated_annealing.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_2_simulated_annealing/ex05_2_simulated_annealing.py) · funções da UC: [`simulated_annealing.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_2_simulated_annealing/simulated_annealing.py)

```text
Reproduz os exemplos do deck 5.2 (simulated annealing).

Função de referência do cap. 5: f(x) = 0.1 x^2 + 3 sin 2x + 2 cos(3.3x + 1)
em X = [-8, 8], mínimo global x* = 2.4681 (f = -4.2381).

1. «Uma execução passo a passo»: x0 = 6, T0 = 3, c = 0.98, sigma = 0.5,
   Nmax = 400, semente 3: T e x_k em k = 30, 100, 200; o salto em k = 315
   (de 4.73 para 3.07, a 3.3 sigma); x = 2.464, n_f = 401.
   E «SA -> método local»: de x_SA, uma pesquisa direta chega a |x - x*| < 1e-6
   com 37 avaliações.
2. «Uma execução não chega: 100 corridas»: a tabela de 6 linhas (sucesso
   |x - x*| < 0.3, sempre de x0 = 6, semente 99): 35, 19, 74, 20, 88, 73 %.
3. «Em duas dimensões: Rastrigin»: de (4, -4), T0 = 20, c = 0.995,
   sigma = 0.5, Nmax = 3000, semente 3: f = 0.068, |x| = 0.02, n_f = 3001;
   30 corridas de pontos aleatórios (semente 11), sucesso f < 1: 97 % com
   c = 0.9, 0.95, 0.99; 100 % com 0.995; 90 % com 0.999.
   E, a título informativo, a chamada do slide «No computador» (sementes 1..30).

Os números aleatórios são os dos scripts das figuras (make_figs_5.py): o
mesmo gerador numpy (default_rng), as mesmas sementes e a mesma ordem de
consumo, por isso os valores dos slides reproduzem-se EXATAMENTE.
Demora alguns segundos (cerca de 10^6 iterações ao todo).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from simulated_annealing import simulated_annealing


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


def rastrigin(x):
    x = np.asarray(x, float)
    return 10 * x.shape[-1] + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=-1)


f1 = lambda x: 0.1 * x**2 + 3 * np.sin(2 * x) + 2 * np.cos(3.3 * x + 1)
f = lambda x: f1(x[0])          # f de um vetor com 1 componente (como no script)
LB, UB = -8.0, 8.0
XS = 2.4680679685               # mínimo global da função de referência

ok = True
print("Otimização — deck 5.2: simulated annealing")

# ------------------------------------------------ 1. Execução passo a passo
x0, T0, c, sigma, Nmax = 6.0, 3.0, 0.98, 0.5, 400
print("\n1. Execução passo a passo: x0 = %g, T0 = %g, c = %g, sigma = %g, Nmax = %d, semente 3"
      % (x0, T0, c, sigma, Nmax))
r = simulated_annealing(f, x0, T0, c, sigma, Nmax, LB, UB, rng=np.random.default_rng(3))
H = r.history                   # [k T aceite f(x') f(x_k) f_best x' x_k]
print("%5s %9s %9s %9s %9s" % ("k", "T", "x_k", "f(x_k)", "f_best"))
for k in (0, 30, 100, 200, 314, 315, 400):
    print("%5d %9.4f %9.4f %9.4f %9.4f" % (k, H[k, 1], H[k, 7], H[k, 4], H[k, 5]))
# saltos: propostas aceites a mais de 1 de distância do ponto corrente
xk = H[:, 7]
saltos = [k for k in range(1, Nmax + 1) if H[k, 2] == 1 and abs(xk[k] - xk[k - 1]) > 1]
kj = saltos[-1]
dz = (xk[kj] - xk[kj - 1]) / sigma
print("último salto aceite: k = %d, de %.2f para %.2f (%.1f sigma)" % (kj, xk[kj - 1], xk[kj], dz))
print("x = %.4f, f(x) = %.4f, x* = %.4f;  n_f = %d, aceites = %d, T final = %.2e"
      % (r.x[0], r.fx, XS, r.nfev, r.nacc, r.T))
c1 = [confere(H[30, 1], 1.7, 1), confere(H[100, 1], 0.4, 1), confere(H[200, 1], 0.05, 2),
      abs(xk[100] - 4.72) < 0.5 and abs(xk[200] - 4.72) < 0.5, confere(H[100, 5], 0.89, 2),
      kj == 315 and confere([xk[314], xk[315]], [4.73, 3.07], 2) and confere(abs(dz), 3.3, 1),
      confere(r.x[0], 2.464, 3), confere(XS, 2.468, 3), r.nfev == 401]
print("  T(30) = 1.7: %s | T(100) = 0.4: %s | T(200) = 0.05: %s | bacia de 4.7 em k = 100, 200: %s |"
      " f = 0.89: %s\n  salto em k = 315 (4.73 -> 3.07, 3.3 sigma): %s | x = 2.464: %s |"
      " x* = 2.468: %s | n_f = 401: %s" % simnao(c1))
ok = ok and all(c1)

# SA -> método local: pesquisa direta 1D (poll {+1, -1}), passo inicial 0.01,
# mantido no sucesso e reduzido a metade no insucesso, até ser <= 1e-6
# (a de make_figs_54.py; não conta f(x_SA), já avaliado pelo SA)
x = float(r.x[0]); fx = f1(x); nf = 0; a = 0.01
while a > 1e-6:
    sucesso = False
    for d in (1, -1):
        y = x + a * d; fy = f1(y); nf += 1
        if fy < fx:
            x, fx, sucesso = y, fy, True
            break
    if not sucesso:
        a /= 2
print("SA -> método local: de %.4f, pesquisa direta: x = %.7f, |x - x*| = %.1e, %d avaliações"
      " (total %d)" % (r.x[0], x, abs(x - XS), nf, r.nfev + nf))
c1 = [abs(x - XS) < 1e-6, nf == 37]
print("  |x - x*| < 1e-6: %s | 37 avaliações: %s" % simnao(c1))
ok = ok and all(c1)

# ------------------------------------------------ 2. 100 corridas
print("\n2. 100 corridas de x0 = 6 (semente 99), sucesso |x - x*| < 0.3")
print("%6s %6s %7s %6s %10s %8s %8s" % ("sigma", "T0", "c", "Nmax", "T_Nmax", "sucesso", "slide"))
linhas = [(0.5, 3.0, 0.98, 400, 35), (0.5, 3.0, 0.90, 400, 19), (0.5, 3.0, 0.995, 400, 74),
          (0.5, 0.3, 0.98, 400, 20), (1.0, 3.0, 0.98, 400, 88), (0.5, 3.0, 0.98, 2000, 73)]
Tslide = [9.3e-4, 1.5e-18, 4.0e-1, 9.3e-5, 9.3e-4, 8.5e-18]
c2 = []
for (sg, T0_, c_, N_, s_), Ts in zip(linhas, Tslide):
    rng = np.random.default_rng(99)
    suc = 0
    for _ in range(100):
        rr = simulated_annealing(f, 6.0, T0_, c_, sg, N_, LB, UB, rng=rng)
        suc += abs(rr.x[0] - XS) < 0.3
    print("%6.1f %6.1f %7.3f %6d %10.1e %7d%% %7d%%" % (sg, T0_, c_, N_, rr.T, suc, s_))
    c2 += [suc == s_, "%.1e" % rr.T == "%.1e" % Ts]     # T_Nmax com 2 algarismos
print("  taxas 35/19/74/20/88/73 %%: %s | T_Nmax: %s"
      % ("sim" if all(c2[0::2]) else "não", "sim" if all(c2[1::2]) else "não"))
ok = ok and all(c2)

# ------------------------------------------------ 3. Rastrigin 2D
print("\n3. Rastrigin 2D em [-5.12, 5.12]^2: T0 = 20, c = 0.995, sigma = 0.5, Nmax = 3000")
r = simulated_annealing(rastrigin, [4.0, -4.0], 20.0, 0.995, 0.5, 3000, -5.12, 5.12,
                        rng=np.random.default_rng(3))
print("de (4, -4), semente 3: x = (%.4f; %.4f), f = %.4f, |x| = %.4f, n_f = %d, aceites = %d"
      % (r.x[0], r.x[1], r.fx, np.linalg.norm(r.x), r.nfev, r.nacc))
c3 = [confere(r.fx, 0.068, 3), confere(np.linalg.norm(r.x), 0.02, 2), r.nfev == 3001]
print("  f = 0.068: %s | |x| = 0.02: %s | n_f = 3001: %s" % simnao(c3))
ok = ok and all(c3)

print("30 corridas de pontos aleatórios (semente 11), sucesso f < 1:")
c3 = []
for c_, s_ in ((0.90, 97), (0.95, 97), (0.99, 97), (0.995, 100), (0.999, 90)):
    rng = np.random.default_rng(11)
    suc = 0
    for _ in range(30):
        x0 = rng.uniform(-5.12, 5.12, 2)
        rr = simulated_annealing(rastrigin, x0, 20.0, c_, 0.5, 3000, -5.12, 5.12, rng=rng)
        suc += rr.fx < 1.0
    print("  c = %.3f: sucesso %3.0f%%  (slide %d%%), T_3000 = %.1e"
          % (c_, 100 * suc / 30, s_, rr.T))
    c3.append(round(100 * suc / 30) == s_)
print("  taxas 97/97/97/100/90 %%: %s" % ("sim" if all(c3) else "não"))
ok = ok and all(c3)

# a chamada do slide «No computador», com default_rng(s) no lugar de rng(s)
# (informativo: o slide não mostra o resultado desta chamada)
fs = np.zeros(30)
for s in range(1, 31):
    rng = np.random.default_rng(s)
    x0 = -5.12 + 10.24 * rng.random(2)
    rr = simulated_annealing(rastrigin, x0, 20, 0.995, 0.5, 3000, -5.12, 5.12, rng=rng)
    fs[s - 1] = rr.fx
q1, q3 = np.percentile(fs, [25, 75])
print("chamada do slide (sementes 1..30): mediana f = %.3f, [Q1; Q3] = [%.3f; %.3f], sucesso %.0f%%,"
      " nfev = %d  (informativo)" % (np.median(fs), q1, q3, 100 * np.mean(fs < 1), rr.nfev))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 5.2: simulated annealing

1. Execução passo a passo: x0 = 6, T0 = 3, c = 0.98, sigma = 0.5, Nmax = 400, semente 3
    k         T       x_k    f(x_k)    f_best
    0    3.0000    6.0000    1.2491    1.2491
   30    1.6698    6.0914    1.3283    1.2490
  100    0.4060    4.7971    0.9284    0.8932
  200    0.0538    4.6047    0.9945    0.8896
  314    0.0054    4.7319    0.8903    0.8895
  315    0.0053    3.0659    0.7308    0.7308
  400    0.0009    2.4642   -4.2379   -4.2379
último salto aceite: k = 315, de 4.73 para 3.07 (-3.3 sigma)
x = 2.4642, f(x) = -4.2379, x* = 2.4681;  n_f = 401, aceites = 124, T final = 9.28e-04
  T(30) = 1.7: sim | T(100) = 0.4: sim | T(200) = 0.05: sim | bacia de 4.7 em k = 100, 200: sim | f = 0.89: sim
  salto em k = 315 (4.73 -> 3.07, 3.3 sigma): sim | x = 2.464: sim | x* = 2.468: sim | n_f = 401: sim
SA -> método local: de 2.4642, pesquisa direta: x = 2.4680674, |x - x*| = 5.2e-07, 37 avaliações (total 438)
  |x - x*| < 1e-6: sim | 37 a

   0.5    3.0   0.980    400    9.3e-04      35%      35%


   0.5    3.0   0.900    400    1.5e-18      19%      19%


   0.5    3.0   0.995    400    4.0e-01      74%      74%


   0.5    0.3   0.980    400    9.3e-05      20%      20%


...

  c = 0.950: sucesso  97%  (slide 97%), T_3000 = 3.0e-66


  c = 0.990: sucesso  97%  (slide 97%), T_3000 = 1.6e-12


  c = 0.995: sucesso 100%  (slide 100%), T_3000 = 5.9e-06


  c = 0.999: sucesso  90%  (slide 90%), T_3000 = 9.9e-01
  taxas 97/97/97/100/90 %: sim


chamada do slide (sementes 1..30): mediana f = 0.028, [Q1; Q3] = [0.012; 0.056], sucesso 97%, nfev = 3001  (informativo)

confere com os slides: sim


## 5.3 Algoritmos genéticos

Exemplo [`ex05_3_genetic_algorithms.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_3_genetic_algorithms/ex05_3_genetic_algorithms.py) · funções da UC: [`genetic_algorithm.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_3_genetic_algorithms/genetic_algorithm.py), [`simulated_annealing.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/05_global_optimization/05_2_simulated_annealing/simulated_annealing.py)

```text
Reproduz os exemplos do deck 5.3 (algoritmos genéticos).

1. «Uma geração à mão»: min x^2 em [-5, 5], 8 bits, N = 6, roleta com
   F = 1/(1+x^2), cruzamento a 1 ponto (pc = 0.8), mutação por bit
   (pm = 0.05); semente 5 do script. População inicial, roleta (xi e pais),
   cortes, descendentes, a mutação e as médias de f: 11.9 -> 10.4 (o melhor
   piora: 0.111 -> 0.323). É o AG binário do slide, não genetic_algorithm
   (que usa codificação real); por isso está escrito aqui, passo a passo.
2. «Rastrigin: explorar ou já convergir?»: N = 40, G = 60, pc = 0.9,
   pm = 0.1, sigma = 0.5, 2 elites (2440 avaliações), semente 2: melhor e
   média nas gerações 0, 5, 20, 60, f < 1e-12 no fim. E «A mesma função,
   várias estratégias»: 30 corridas (semente 21, a amostra do script, com o
   SA do 5.2 intercalado): AG 100 % com n_f = 2440, SA 100 % com n_f = 3001.
   E, a título informativo, a chamada do slide «No computador» (sementes 1..30).
3. «O que fazem os parâmetros?»: função de referência do cap. 5, 300 corridas
   (semente 11), sucesso |x - x*| < 0.3: referência 75 %, s = 6 61 %, pc = 0
   60 %, N = 6 34 %, N = 60 96 %; pm = 0 76 %, pm = 0.8 78 %; mesmo orçamento
   n_f = 420: N = 60 (G = 6) 96 %, N = 6 (G = 69) 46 %.

Os números aleatórios são os dos scripts das figuras (make_figs_5.py e
make_figs_54.py): o mesmo gerador numpy (default_rng), as mesmas sementes e a
mesma ordem de consumo, por isso os valores dos slides reproduzem-se
EXATAMENTE. Uma única diferença de ordem, na parte 3, é tratada com um
adaptador explicado abaixo (OrdemMakeFigs54).
Demora cerca de 20 s (2700 corridas do AG na parte 3). Com COMPLETO = False
a parte 3 faz só 30 corridas por linha e mostra as taxas sem as conferir
(os valores exatos do slide só saem com COMPLETO = True).
Usa simulated_annealing da pasta 05_2_simulated_annealing (parte 2).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [3]:
import numpy as np

from genetic_algorithm import genetic_algorithm
from simulated_annealing import simulated_annealing  # (5.2)

COMPLETO = True      # False: parte 3 com 30 corridas por linha, sem conferir as taxas


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


def rastrigin(x):
    x = np.asarray(x, float)
    return 10 * x.shape[-1] + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=-1)


ok = True
print("Otimização — deck 5.3: algoritmos genéticos")

# ------------------------------------------------ 1. Uma geração à mão
print("\n1. Uma geração à mão: min x^2 em [-5, 5], 8 bits, N = 6, roleta, pc = 0.8, pm = 0.05 (semente 5)")
L, U, nb, N, pc, pm = -5.0, 5.0, 8, 6, 0.8, 0.05
decode = lambda b: L + (U - L) * int("".join(str(v) for v in b), 2) / (2**nb - 1)
cadeia = lambda b: "".join(str(v) for v in b)
rng = np.random.default_rng(5)
pop = rng.integers(0, 2, (N, nb))                      # população inicial
x = np.array([decode(b) for b in pop]); fx = x**2
Fa = 1 / (1 + fx); p = Fa / Fa.sum(); acum = np.cumsum(p)
print("%2s %9s %7s %7s %6s %6s %6s" % ("i", "cromos.", "x", "f", "F", "p_i", "acum."))
for i in range(N):
    print("%2d %9s %7.3f %7.3f %6.3f %6.3f %6.3f" % (i + 1, cadeia(pop[i]), x[i], fx[i], Fa[i], p[i], acum[i]))
xi = rng.random(N); pais = np.searchsorted(acum, xi)  # roleta: 1.º i com acumulada >= xi
print("roleta: xi =", " ".join("%.3f" % v for v in xi), "-> pais", " ".join(str(v + 1) for v in pais))
rc = rng.random(N // 2); cortes = rng.integers(1, nb, N // 2)
filhos = []
for j in range(N // 2):
    a, b = pop[pais[2 * j]], pop[pais[2 * j + 1]]
    if rc[j] < pc:                                     # cruzamento a 1 ponto
        cj = cortes[j]
        c1 = np.concatenate([a[:cj], b[cj:]]); c2 = np.concatenate([b[:cj], a[cj:]])
        nota = "corte após bit %d" % cj
    else:
        c1, c2 = a.copy(), b.copy(); nota = "sem cruzamento"
    filhos += [c1, c2]
    print("pais %d, %d: %s, %s  %s  -> %s, %s" % (pais[2 * j] + 1, pais[2 * j + 1] + 1, cadeia(a),
                                                cadeia(b), nota, cadeia(c1), cadeia(c2)))
filhos = np.array(filhos)
mut = rng.random(filhos.shape) < pm                    # mutação: inverter o bit
filhos_m = np.where(mut, 1 - filhos, filhos)
x1 = np.array([decode(b) for b in filhos_m]); fx1 = x1**2
print("mutações: %d (descendente %s, bit %s)" % (mut.sum(), ", ".join(str(i + 1) for i in np.argwhere(mut)[:, 0]),
                                                ", ".join(str(j + 1) for j in np.argwhere(mut)[:, 1])))
for i in range(N):
    print("%2d %9s -> %9s %7.3f %7.3f" % (i + 1, cadeia(filhos[i]), cadeia(filhos_m[i]), x1[i], fx1[i]))
print("média de f: %.1f -> %.1f;  melhor: %.3f -> %.3f" % (fx.mean(), fx1.mean(), fx.min(), fx1.min()))
c1 = [[cadeia(b) for b in pop] == ["11010110", "10001000", "00010110", "00010111", "00101110", "11100100"],
      confere(x, [3.392, 0.333, -4.137, -4.098, -3.196, 3.941], 3),
      confere(fx, [11.507, 0.111, 17.117, 16.794, 10.215, 15.533], 3),
      confere(Fa, [0.080, 0.900, 0.055, 0.056, 0.089, 0.060], 3),
      confere(p, [0.064, 0.725, 0.044, 0.045, 0.072, 0.049], 3),
      confere(acum, [0.064, 0.790, 0.834, 0.879, 0.951, 1.000], 3),
      confere(xi, [0.679, 0.870, 0.227, 0.895, 0.872, 0.019], 3) and list(pais + 1) == [2, 4, 2, 5, 4, 1],
      list(cortes) == [3, 4, 7] and all(rc < pc),
      [cadeia(b) for b in filhos_m] == ["10010111", "00001100", "10001110", "00101000", "00010110", "11010111"]
      and mut.sum() == 1,
      confere(x1, [0.922, -4.529, 0.569, -3.431, -4.137, 3.431], 3),
      confere(fx1, [0.849, 20.516, 0.323, 11.774, 17.117, 11.774], 3),
      confere([fx.mean(), fx1.mean()], [11.9, 10.4], 1) and confere([fx.min(), fx1.min()], [0.111, 0.323], 3)]
print("  cromossomas: %s | x: %s | f: %s | F: %s | p_i: %s | acumulada: %s | roleta: %s |\n"
      "  cortes 3, 4, 7: %s | descendentes e mutação: %s | x: %s | f: %s | médias 11.9 -> 10.4: %s" % simnao(c1))
ok = ok and all(c1)

# ------------------------------------------------ 2. Rastrigin 2D
print("\n2. Rastrigin 2D em [-5.12, 5.12]^2: N = 40, G = 60, pc = 0.9, pm = 0.1, sigma = 0.5, 2 elites")
r = genetic_algorithm(rastrigin, 2, -5.12, 5.12, 40, 60, 0.9, 0.1, 0.5, 2, rng=np.random.default_rng(2))
H = r.history                                          # [t melhor média x]
for t in (0, 5, 20, 60):
    print("  geração %2d: melhor f = %.3g, média = %.1f" % (t, H[t, 1], H[t, 2]))
print("semente 2: x = (%.1e; %.1e), f = %.2e, n_f = %d" % (r.x[0], r.x[1], r.fx, r.nfev))
c2 = [H[20, 1] < 1, r.fx < 1e-12, r.nfev == 2440,
      confere(H[[0, 5, 20, 60], 2], [31.3, 15.2, 2.0, 1.4], 1) and confere(H[[0, 5], 1], [6.479, 0.581], 3)]
print("  na bacia do global na geração 20: %s | f < 1e-12: %s | n_f = 2440: %s |"
      " melhor/média da figura: %s" % simnao(c2))
ok = ok and all(c2)

# a amostra de 30 pontos do script (semente 21): SA e AG intercalados no mesmo gerador
rng = np.random.default_rng(21)
fsa = np.zeros(30); fag = np.zeros(30)
for k in range(30):
    x0 = rng.uniform(-5.12, 5.12, 2)
    rs = simulated_annealing(rastrigin, x0, 20.0, 0.995, 0.5, 3000, -5.12, 5.12, rng=rng)
    ra = genetic_algorithm(rastrigin, 2, -5.12, 5.12, 40, 60, 0.9, 0.1, 0.5, 2, rng=rng)
    fsa[k] = rs.fx; fag[k] = ra.fx
print("30 corridas (semente 21), sucesso f < 1: AG %.0f%% (n_f = %d), SA %.0f%% (n_f = %d);"
      " medianas f: AG %.1e, SA %.3f" % (100 * np.mean(fag < 1), ra.nfev, 100 * np.mean(fsa < 1), rs.nfev,
                                         np.median(fag), np.median(fsa)))
c2 = [np.all(fag < 1), ra.nfev == 2440, np.all(fsa < 1), rs.nfev == 3001]
print("  AG 100 %%: %s | 2440: %s | SA 100 %%: %s | 3001: %s" % simnao(c2))
ok = ok and all(c2)

# a chamada do slide «No computador», com default_rng(s) no lugar de rng(s) (informativo)
fs = np.zeros(30)
for s in range(1, 31):
    rr = genetic_algorithm(rastrigin, 2, -5.12, 5.12, 40, 60, 0.9, 0.1, 0.5, 2,
                           rng=np.random.default_rng(s))
    fs[s - 1] = rr.fx
q1, q3 = np.percentile(fs, [25, 75])
print("chamada do slide (sementes 1..30): mediana f = %.1e, [Q1; Q3] = [%.1e; %.1e], sucesso %.0f%%,"
      " nfev = %d  (informativo)" % (np.median(fs), q1, q3, 100 * np.mean(fs < 1), rr.nfev))

# ------------------------------------------------ 3. O que fazem os parâmetros?
f1 = lambda x: 0.1 * x**2 + 3 * np.sin(2 * x) + 2 * np.cos(3.3 * x + 1)
# f de um vetor com 1 componente, avaliada como vetor (o script avalia a
# população de uma vez; assim os valores coincidem bit a bit)
f = lambda x: f1(x)[0]
LB, UB = -8.0, 8.0
XS = 2.4680679685


class OrdemMakeFigs54:
    """Adaptador do gerador, só para reproduzir make_figs_54.py bit a bit.

    genetic_algorithm sorteia os candidatos do torneio como uma matriz s x N
    (linha r = r-ésimo candidato de cada pai: i1, i2 do slide); make_figs_54.py
    sorteia-os como N x s (linha = pai). A distribuição é a mesma; só muda a
    ordem em que os números saem do gerador. Este adaptador faz o sorteio na
    ordem do script e devolve-o transposto; tudo o resto passa ao gerador.
    """
    def __init__(self, rng):
        self.rng = rng

    def integers(self, low, high, size):
        s, N = size
        return self.rng.integers(low, high, (N, s)).T

    def __getattr__(self, nome):
        return getattr(self.rng, nome)


def taxa(R, N=20, G=20, pc=0.9, pm=0.1, sigma=0.5, s=2):
    rng = OrdemMakeFigs54(np.random.default_rng(11))
    suc = 0
    for _ in range(R):
        rr = genetic_algorithm(f, 1, LB, UB, N, G, pc, pm, sigma, 2, s=s, rng=rng)
        suc += abs(rr.x[0] - XS) < 0.3
    return 100 * suc / R, rr.nfev


R = 300 if COMPLETO else 30
print("\n3. Função de referência, %d corridas (semente 11), sucesso |x - x*| < 0.3%s"
      % (R, "" if COMPLETO else "  [COMPLETO = False: taxas só indicativas]"))
linhas = [("referência: N = 20, s = 2, pm = 0.1", {}, 75, 420),
          ("pressão forte: s = 6", dict(s=6), 61, 420),
          ("sem cruzamento: pc = 0", dict(pc=0.0), 60, 420),
          ("população pequena: N = 6", dict(N=6), 34, 126),
          ("população maior: N = 60", dict(N=60), 96, 1260),
          ("sem mutação: pm = 0", dict(pm=0.0), 76, 420),
          ("mutação alta: pm = 0.8", dict(pm=0.8), 78, 420),
          ("mesmo orçamento: N = 60, G = 6", dict(N=60, G=6), 96, 420),
          ("mesmo orçamento: N = 6, G = 69", dict(N=6, G=69), 46, 420)]
c3t = []; c3n = []
for nome, kw, s_, nf_ in linhas:
    tx, nf = taxa(R, **kw)
    print("  %-36s sucesso %5.1f%%  (slide %d%%)  n_f = %4d (slide %d)" % (nome, tx, s_, nf, nf_))
    c3t.append(round(tx) == s_); c3n.append(nf == nf_)
if COMPLETO:
    print("  taxas 75/61/60/34/96, 76/78, 96/46 %%: %s | n_f: %s" % simnao([all(c3t), all(c3n)]))
    ok = ok and all(c3t) and all(c3n)
else:
    print("  n_f: %s (taxas não conferidas: só com COMPLETO = True, 300 corridas)" % simnao([all(c3n)]))
    ok = ok and all(c3n)

print("\nconfere com os slides: %s%s" % ("sim" if ok else "não",
                                        "" if COMPLETO else " (sem as taxas da parte 3)"))


Otimização — deck 5.3: algoritmos genéticos

1. Uma geração à mão: min x^2 em [-5, 5], 8 bits, N = 6, roleta, pc = 0.8, pm = 0.05 (semente 5)
 i   cromos.       x       f      F    p_i  acum.
 1  11010110   3.392  11.507  0.080  0.064  0.064
 2  10001000   0.333   0.111  0.900  0.725  0.790
 3  00010110  -4.137  17.117  0.055  0.044  0.834
 4  00010111  -4.098  16.794  0.056  0.045  0.879
 5  00101110  -3.196  10.215  0.089  0.072  0.951
 6  11100100   3.941  15.533  0.060  0.049  1.000
roleta: xi = 0.679 0.870 0.227 0.895 0.872 0.019 -> pais 2 4 2 5 4 1
pais 2, 4: 10001000, 00010111  corte após bit 3  -> 10010111, 00001000
pais 2, 5: 10001000, 00101110  corte após bit 4  -> 10001110, 00101000
pais 4, 1: 00010111, 11010110  corte após bit 7  -> 00010110, 11010111
mutações: 1 (descendente 2, bit 6)
 1  10010111 ->  10010111   0.922   0.849
 2  00001000 ->  00001100  -4.529  20.516
 3  10001110 ->  10001110   0.569   0.323
 4  00101000 ->  00101000  -3.431  11.774
 5  00010110 ->  000101

30 corridas (semente 21), sucesso f < 1: AG 100% (n_f = 2440), SA 100% (n_f = 3001); medianas f: AG 2.0e-10, SA 0.030
  AG 100 %: sim | 2440: sim | SA 100 %: sim | 3001: sim


chamada do slide (sementes 1..30): mediana f = 2.4e-11, [Q1; Q3] = [6.2e-13; 1.6e-08], sucesso 100%, nfev = 2440  (informativo)

3. Função de referência, 300 corridas (semente 11), sucesso |x - x*| < 0.3


  referência: N = 20, s = 2, pm = 0.1  sucesso  74.7%  (slide 75%)  n_f =  420 (slide 420)


  pressão forte: s = 6                 sucesso  60.7%  (slide 61%)  n_f =  420 (slide 420)


...

  população maior: N = 60              sucesso  96.3%  (slide 96%)  n_f = 1260 (slide 1260)


  sem mutação: pm = 0                  sucesso  76.3%  (slide 76%)  n_f =  420 (slide 420)


  mutação alta: pm = 0.8               sucesso  78.0%  (slide 78%)  n_f =  420 (slide 420)


  mesmo orçamento: N = 60, G = 6       sucesso  96.3%  (slide 96%)  n_f =  420 (slide 420)


  mesmo orçamento: N = 6, G = 69       sucesso  45.7%  (slide 46%)  n_f =  420 (slide 420)
  taxas 75/61/60/34/96, 76/78, 96/46 %: sim | n_f: sim

confere com os slides: sim
